# 🤖 Fase 4 — Implementando um Agente com Design Patterns
## Da Teoria à Prática: Agente LLM em Java com Padrões Arquiteturais

---
**Roteiro-Desafio-ES · Fatec SCS · ADS 4º Semestre · 2026**  
**Grupo:** Isaac Gomes **Data:** 18/05/2026

---

### 🎯 Objetivo desta fase
Implementar em Java um **mini-agente baseado em LLM** que aplique pelo menos 5 padrões do catálogo CSIRO usando a base GoF e arquitetura Hexagonal/Clean.

### 📚 O que você vai aprender
- Integrar uma API de LLM (OpenAI) em Java
- Aplicar múltiplos padrões simultaneamente em um sistema real
- Compor padrões em uma arquitetura coerente
- Implementar RAG, Reflection e Tool Registry na prática

### 🏗️ Padrões a implementar
1. **Passive Goal Creator** (Factory Method)
2. **RAG** (Strategy + Proxy)
3. **Single-Path Plan Generator** (Builder)
4. **Self-Reflection** (Template Method + Chain of Responsibility)
5. **Tool/Agent Registry** (Service Locator)

---


## 📦 Passo 1 — Estrutura do Projeto

In [ ]:
// ══ ESTRUTURA HEXAGONAL DO AGENTE ══
System.out.println("📁 Estrutura do projeto:");
System.out.println("└── agent-system/");
System.out.println("    ├── domain/              ← HEXÁGONO CENTRAL");
System.out.println("    │   ├── model/");
System.out.println("    │   │   ├── Goal.java         (Entity)");
System.out.println("    │   │   ├── Plan.java          (Entity)");
System.out.println("    │   │   └── ReflectionResult.java (Value Object)");
System.out.println("    │   ├── port/");
System.out.println("    │   │   ├── in/");
System.out.println("    │   │   │   └── AgentService.java    (Driving Port)");
System.out.println("    │   │   └── out/");
System.out.println("    │   │       ├── LLMGateway.java      (Driven Port)");
System.out.println("    │   │       ├── KnowledgeStore.java  (Driven Port)");
System.out.println("    │   │       └── ToolRegistry.java    (Driven Port)");
System.out.println("    │   └── service/");
System.out.println("    │       └── AgentServiceImpl.java  (Domain Service)");
System.out.println("    ├── adapter/             ← ADAPTERS");
System.out.println("    │   ├── in/");
System.out.println("    │   │   └── CLIAdapter.java        (Driving Adapter)");
System.out.println("    │   └── out/");
System.out.println("    │       ├── OpenAIGateway.java     (Driven Adapter)");
System.out.println("    │       ├── InMemoryKnowledge.java (Driven Adapter)");
System.out.println("    │       └── SimpleToolRegistry.java(Driven Adapter)");
System.out.println("    └── Main.java            ← COMPOSIÇÃO");


## 🔹 Passo 2 — Entities e Ports (Domínio)

In [ ]:
// ══ DOMAIN: ENTITIES ══
import java.util.*;
import java.time.*;

class Goal {
    private final String description;
    private final LocalDateTime createdAt;
    
    public Goal(String description) {
        this.description = Objects.requireNonNull(description);
        this.createdAt = LocalDateTime.now();
    }
    public String getDescription() { return description; }
    public LocalDateTime getCreatedAt() { return createdAt; }
}

class Plan {
    private final Goal goal;
    private final List<String> steps;
    private boolean reflected;
    private String reflectionNote;
    
    public Plan(Goal goal, List<String> steps) {
        this.goal = goal;
        this.steps = new ArrayList<>(steps);
        this.reflected = false;
    }
    public Goal getGoal() { return goal; }
    public List<String> getSteps() { return Collections.unmodifiableList(steps); }
    public void markReflected(String note) {
        this.reflected = true;
        this.reflectionNote = note;
    }
    public boolean isReflected() { return reflected; }
    public String getReflectionNote() { return reflectionNote; }
}

// ══ DOMAIN: PORTS (interfaces) ══

// Driven Port — LLM
interface LLMGateway {
    String complete(String prompt);
}

// Driven Port — Base de Conhecimento (para RAG)
interface KnowledgeStore {
    List<String> search(String query, int topK);
}

// Driven Port — Registro de Ferramentas
interface ToolRegistry {
    void register(String name, ToolInterface tool);
    ToolInterface lookup(String name);
    List<String> listTools();
}

// Interface unificada de ferramenta (Agent Adapter pattern)
interface ToolInterface {
    String execute(String input);
}

// Driving Port — serviço do agente
interface AgentService {
    Plan processGoal(String userInput);
}

System.out.println("✅ Domínio definido: Entities + Ports");
System.out.println("   Nenhuma dependência externa — apenas interfaces!");


## 🔹 Passo 3 — Domain Service (orquestração dos padrões)

In [ ]:
// ══ DOMAIN SERVICE — Orquestra os 5 padrões ══

class AgentServiceImpl implements AgentService {
    private final LLMGateway llm;
    private final KnowledgeStore knowledge;
    private final ToolRegistry tools;
    
    // Dependency Injection via construtor
    public AgentServiceImpl(LLMGateway llm, KnowledgeStore knowledge, ToolRegistry tools) {
        this.llm = llm;
        this.knowledge = knowledge;
        this.tools = tools;
    }
    
    public Plan processGoal(String userInput) {
        System.out.println("\n🤖 Agente iniciado...");
        
        // ── PADRÃO 1: Passive Goal Creator (Factory Method) ──
        Goal goal = createGoal(userInput);
        System.out.println("📝 [Passive Goal Creator] Objetivo: " + goal.getDescription());
        
        // ── PADRÃO 2: RAG (Strategy) ──
        List<String> context = retrieveContext(goal.getDescription());
        System.out.println("📚 [RAG] Contexto recuperado: " + context.size() + " documentos");
        
        // ── PADRÃO 3: Single-Path Plan Generator (Builder) ──
        Plan plan = generatePlan(goal, context);
        System.out.println("📋 [Plan Generator] Plano com " + plan.getSteps().size() + " passos");
        
        // ── PADRÃO 4: Self-Reflection (Template Method) ──
        Plan reflectedPlan = selfReflect(plan);
        System.out.println("🪞 [Self-Reflection] " + reflectedPlan.getReflectionNote());
        
        // ── PADRÃO 5: Tool Registry (Service Locator) ──
        listAvailableTools();
        
        return reflectedPlan;
    }
    
    // ── Padrão 1: Goal Creation ──
    private Goal createGoal(String input) {
        // Passive: analisa o prompt direto do usuário
        String refined = llm.complete("Reformule como objetivo claro: " + input);
        return new Goal(refined);
    }
    
    // ── Padrão 2: RAG ──
    private List<String> retrieveContext(String query) {
        List<String> docs = knowledge.search(query, 3);
        return docs;
    }
    
    // ── Padrão 3: Plan Generation (Builder pattern interno) ──
    private Plan generatePlan(Goal goal, List<String> context) {
        String contextStr = String.join("\n", context);
        String prompt = "Contexto:\n" + contextStr + 
                        "\nObjetivo: " + goal.getDescription() +
                        "\nGere um plano em passos numerados:";
        String response = llm.complete(prompt);
        List<String> steps = List.of(response.split("\n"));
        return new Plan(goal, steps);
    }
    
    // ── Padrão 4: Self-Reflection ──
    private Plan selfReflect(Plan plan) {
        String stepsStr = String.join("; ", plan.getSteps());
        String prompt = "Avalie este plano e sugira melhorias: " + stepsStr;
        String reflection = llm.complete(prompt);
        plan.markReflected(reflection);
        return plan;
    }
    
    // ── Padrão 5: Tool Registry ──
    private void listAvailableTools() {
        List<String> available = tools.listTools();
        System.out.println("🔧 [Tool Registry] Ferramentas disponíveis: " + available);
    }
}

System.out.println("✅ Domain Service definido com 5 padrões CSIRO!");


## 🔹 Passo 4 — Adapters (implementações concretas)

In [ ]:
// ══ DRIVEN ADAPTERS ══

// Adapter para LLM (simulado — substitua por chamada real à API OpenAI)
class SimulatedLLMGateway implements LLMGateway {
    public String complete(String prompt) {
        // Simulação — em produção, chame a API OpenAI via HTTP
        if (prompt.contains("Reformule")) {
            return "Desenvolver um sistema de FAQ inteligente com busca semântica";
        } else if (prompt.contains("plano")) {
            return "1. Definir domínio e entidades\n2. Configurar base de conhecimento\n" +
                   "3. Implementar RAG\n4. Criar interface de busca\n5. Testar com dados reais";
        } else if (prompt.contains("Avalie")) {
            return "✅ Plano coerente. Sugestão: adicionar passo de monitoramento pós-deploy.";
        }
        return "[LLM Response para: " + prompt.substring(0, Math.min(50, prompt.length())) + "...]";
    }
}

// Adapter para Knowledge Store (In-Memory)
class InMemoryKnowledge implements KnowledgeStore {
    private List<String> docs = List.of(
        "Design Patterns são soluções reutilizáveis para problemas recorrentes em software.",
        "RAG combina retrieval de documentos com geração de texto por LLMs.",
        "Clean Architecture separa regras de negócio das dependências externas.",
        "O catálogo CSIRO define 18 padrões para agentes baseados em Foundation Models.",
        "Hexagonal Architecture usa Ports e Adapters para isolar o domínio."
    );
    
    public List<String> search(String query, int topK) {
        // Busca simplificada por keyword match
        return docs.stream()
            .filter(d -> {
                String lower = d.toLowerCase();
                return Arrays.stream(query.toLowerCase().split("\\s+"))
                    .anyMatch(lower::contains);
            })
            .limit(topK)
            .toList();
    }
}

// Adapter para Tool Registry
class SimpleToolRegistry implements ToolRegistry {
    private Map<String, ToolInterface> registry = new LinkedHashMap<>();
    
    public void register(String name, ToolInterface tool) {
        registry.put(name, tool);
    }
    
    public ToolInterface lookup(String name) {
        ToolInterface tool = registry.get(name);
        if (tool == null) throw new RuntimeException("Ferramenta não encontrada: " + name);
        return tool;
    }
    
    public List<String> listTools() {
        return new ArrayList<>(registry.keySet());
    }
}

System.out.println("✅ Adapters implementados!");


## 🚀 Passo 5 — Composição e Execução

In [ ]:
// ══ COMPOSIÇÃO (Main) — "Wiring" dos adapters nos ports ══

// 1. Cria adapters
LLMGateway llm = new SimulatedLLMGateway();
KnowledgeStore kb = new InMemoryKnowledge();
SimpleToolRegistry registry = new SimpleToolRegistry();

// 2. Registra ferramentas (Tool Registry pattern)
registry.register("web-search", input -> "🌐 Resultado de busca para: " + input);
registry.register("calculator", input -> "🧮 Resultado: " + input + " = 42");
registry.register("code-executor", input -> "💻 Código executado: " + input);

// 3. Injeta no domain service
AgentService agent = new AgentServiceImpl(llm, kb, registry);

// 4. Executa!
System.out.println("═══════════════════════════════════════════");
System.out.println("       🤖 MINI-AGENTE LLM EM JAVA");
System.out.println("       5 Padrões CSIRO · Arq. Hexagonal");
System.out.println("═══════════════════════════════════════════");

Plan result = agent.processGoal("Quero criar um FAQ inteligente para minha empresa");

System.out.println("\n═══════════════════════════════════════════");
System.out.println("📋 RESULTADO FINAL:");
System.out.println("   Objetivo: " + result.getGoal().getDescription());
System.out.println("   Passos:");
int k = 1;
for (String step : result.getSteps()) {
    System.out.println("     " + k++ + ". " + step);
}
System.out.println("   Refletido? " + result.isReflected());
System.out.println("   Nota: " + result.getReflectionNote());
System.out.println("═══════════════════════════════════════════");


## 📝 Avaliação — Fase 4

**Q1.** Na implementação do agente, o `AgentServiceImpl` conhece as classes concretas dos adapters?  
( ) Sim, ele instancia diretamente  
(X) Não, ele recebe apenas interfaces (Ports) via Dependency Injection  
( ) Sim, mas apenas em testes  
( ) Depende da configuração

**Q2.** O Passive Goal Creator do agente usa o LLM para:  
( ) Treinar um modelo novo  (X) Reformular o input do usuário como objetivo claro  
( ) Classificar a intenção do usuário  ( ) Traduzir o objetivo para inglês

**Q3.** No padrão RAG implementado, qual é o papel do `KnowledgeStore`?  
( ) Armazenar os prompts enviados ao LLM  
(X) Fornecer documentos relevantes para aumentar o contexto do prompt  
( ) Salvar os planos gerados  
( ) Registrar ferramentas disponíveis

**Q4.** O Self-Reflection do agente funciona por:  
( ) Treinar novamente o modelo com novos dados  
(X) Enviar o plano de volta ao LLM pedindo avaliação e sugestões  
( ) Comparar o plano com planos anteriores em banco de dados  
( ) Solicitar aprovação do usuário antes de prosseguir

**Q5.** O `SimpleToolRegistry` implementa qual pattern?  
( ) Factory Method  ( ) Observer  (X) Service Locator  ( ) Singleton

**Q6.** Se quisermos trocar de OpenAI para Anthropic Claude, o que precisamos alterar?  
( ) O `AgentServiceImpl` inteiro  
(X) Apenas o adapter que implementa `LLMGateway`  
( ) A interface `LLMGateway` e todos os seus consumidores  
( ) O `Main.java` e todos os testes

**Q7.** O método `processGoal` orquestra os padrões em qual ordem?  
( ) RAG → Goal → Plan → Reflect → Tools  
(X) Goal → RAG → Plan → Reflect → Tools  
( ) Plan → Goal → Reflect → RAG → Tools  
( ) Tools → Goal → RAG → Plan → Reflect

**Q8.** Qual princípio SOLID é violado se o `AgentServiceImpl` instanciar diretamente `new SimulatedLLMGateway()`?  
( ) SRP  ( ) OCP  (X) DIP  ( ) ISP

**Q9.** Em uma versão real do agente, o `InMemoryKnowledge` seria substituído por:  
( ) Outro Singleton  
(X) Um adapter que acessa um banco vetorial (ex: Pinecone, FAISS)  
( ) Uma classe abstrata com métodos estáticos  
( ) Um enum com constantes de documentos

**Q10.** Quantos padrões do catálogo CSIRO foram implementados nesta fase?  
( ) 3  (X) 5  ( ) 7  ( ) 18
